# Perturbation Recall@k — held-out cross-set

For each well in the held-out validation group, retrieve the *k* nearest
neighbours (cosine on the Harmony-corrected PCA embedding) from the
train-time groups, then check whether any neighbour shares the well's
perturbation. Macro-average over perturbations.

This is the cross-set version of recall@k; the older all-vs-all variant
is omitted because every well in the gallery was seen during training,
which inflates retrieval through memorisation rather than measuring
generalisation.


In [ ]:
from __future__ import annotations

from pathlib import Path

import anndata as ad
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.preprocessing import normalize

# Reproducibility: pin numpy global seed to match upstream pipeline.
RANDOM_SEED: int = 42
np.random.seed(RANDOM_SEED)


## Configuration

In [ ]:
EMBED_DIR = Path("../data")  # run notebooks from evals/<eval_name>/

EMBEDDING_KEYS = ("X_pca", "X_pca_harmony")  # pre- and post-Harmony
PERTURBATION_COL = "perturbation"
CONTROL_COL = "is_control"
K_MAX = 10

# Held-out group: queries come from this row subset, gallery from the rest.
# Reproduces ImageMoleculeDataModule's group-wise split (val_frac=0.1, seed=42).
HELD_OUT_COL = 'batch'
HELD_OUT_VALUE = 'plate_4'


## Recall@k

In [ ]:
def compute_held_out_recall(
    adata: ad.AnnData,
    query_mask: np.ndarray,
    perturbation_col: str = "perturbation",
    control_col: str = "is_control",
    embedding_key: str | None = "X_pca_harmony",
    k_max: int = 10,
    batch_size: int = 4096,
) -> pd.DataFrame:
    """Recall@k where queries come from query_mask, candidates from the rest.

    Drops controls before computing similarity; query_mask is filtered to
    the same row order. Returns one row per perturbation (n_query_wells >= 1).
    """
    not_ctrl = adata.obs[control_col].astype(str) != "True"
    adata = adata[not_ctrl].copy()
    query_mask = np.asarray(query_mask)[not_ctrl.values]

    if embedding_key is not None and embedding_key in adata.obsm:
        X = np.asarray(adata.obsm[embedding_key])
    else:
        X = np.asarray(adata.X)
        if hasattr(X, "toarray"):
            X = X.toarray()

    X = normalize(X.astype(np.float64), norm="l2")
    labels = np.array([str(x) for x in adata.obs[perturbation_col]])

    q_idx = np.where(query_mask)[0]
    c_idx = np.where(~query_mask)[0]
    if q_idx.size == 0:
        raise ValueError("query_mask selected zero non-control wells")
    if c_idx.size < k_max:
        raise ValueError(f"gallery has {c_idx.size} wells but k_max={k_max}")

    X_q = X[q_idx]
    X_c = X[c_idx]
    labels_q = labels[q_idx]
    labels_c = labels[c_idx]

    n_q = len(q_idx)
    cumulative_hits = np.zeros((n_q, k_max), dtype=bool)

    for start in range(0, n_q, batch_size):
        end = min(start + batch_size, n_q)
        sim = X_q[start:end] @ X_c.T
        top_k_idx = np.argpartition(-sim, kth=k_max, axis=1)[:, :k_max]
        for i in range(end - start):
            order = np.argsort(-sim[i, top_k_idx[i]])
            top_k_idx[i] = top_k_idx[i][order]
        top_k_labels = labels_c[top_k_idx]
        hits = top_k_labels == labels_q[start:end, None]
        cumulative_hits[start:end] = np.cumsum(hits, axis=1) > 0

    rows = []
    for pert in np.unique(labels_q):
        pert_mask = labels_q == pert
        n_wells = int(pert_mask.sum())
        if n_wells < 1:
            continue
        recall_k = cumulative_hits[pert_mask].mean(axis=0)
        row = {"perturbation": pert, "n_query_wells": n_wells}
        for k in range(1, k_max + 1):
            row[f"R@{k}"] = recall_k[k - 1]
        rows.append(row)
    return pd.DataFrame(rows)


## Configurations to evaluate

In [ ]:
GROUPS: dict[str, dict[str, str]] = {
    'Rxrx3C — DINO': {
        'C': '25_Rxrx3C_DINO_ESM2_C_aggregated.h5ad',
        'CD': '26_Rxrx3C_DINO_ESM2_CD_aggregated.h5ad',
        'S': '27_Rxrx3C_DINO_ESM2_S_aggregated.h5ad',
        'SD': '28_Rxrx3C_DINO_ESM2_SD_aggregated.h5ad',
    },
    'Rxrx3C — OpenPhenom': {
        'C': '29_Rxrx3C_OpenPhenom_ESM2_C_aggregated.h5ad',
        'CD': '30_Rxrx3C_OpenPhenom_ESM2_CD_aggregated.h5ad',
        'S': '31_Rxrx3C_OpenPhenom_ESM2_S_aggregated.h5ad',
        'SD': '32_Rxrx3C_OpenPhenom_ESM2_SD_aggregated.h5ad',
    },
    'Rxrx3C — SubCell': {
        'C': '33_Rxrx3C_SubCell_ESM2_C_aggregated.h5ad',
        'CD': '34_Rxrx3C_SubCell_ESM2_CD_aggregated.h5ad',
        'S': '35_Rxrx3C_SubCell_ESM2_S_aggregated.h5ad',
        'SD': '36_Rxrx3C_SubCell_ESM2_SD_aggregated.h5ad',
    },
}

## Run

In [ ]:
# group_results[group_name][view][embedding_key] = per-perturbation DataFrame
group_results: dict[str, dict[str, dict[str, pd.DataFrame]]] = {}

for group_name, view_files in GROUPS.items():
    print(f"=== {group_name} ===")
    group_results[group_name] = {}
    for view, fname in view_files.items():
        path = EMBED_DIR / fname
        adata = ad.read_h5ad(path)
        query_mask = (adata.obs[HELD_OUT_COL].astype(str) == HELD_OUT_VALUE).values
        group_results[group_name][view] = {}
        for embedding_key in EMBEDDING_KEYS:
            df = compute_held_out_recall(
                adata,
                query_mask=query_mask,
                perturbation_col=PERTURBATION_COL,
                control_col=CONTROL_COL,
                embedding_key=embedding_key,
                k_max=K_MAX,
            )
            group_results[group_name][view][embedding_key] = df
            tag = "pre " if embedding_key == "X_pca" else "post"
            print(
                f"  {view:<3} [{tag}]  n_query={int(query_mask.sum()):>5}  "
                f"n_gallery={int((~query_mask).sum()):>5}  "
                f"perts={len(df):>4}  "
                f"R@1={df['R@1'].mean():.4f}  R@5={df['R@5'].mean():.4f}  R@10={df['R@10'].mean():.4f}"
            )
    print()


## Summary

In [ ]:
summary_rows = []
for group_name, view_results in group_results.items():
    for view, key_results in view_results.items():
        for embedding_key, df in key_results.items():
            row = {
                "group": group_name,
                "view": view,
                "embedding": "pre_harmony" if embedding_key == "X_pca" else "post_harmony",
                "n_perturbations": len(df),
                "n_query_wells": int(df["n_query_wells"].sum()),
            }
            for k in range(1, K_MAX + 1):
                row[f"R@{k}"] = df[f"R@{k}"].mean()
            summary_rows.append(row)

summary = pd.DataFrame(summary_rows)

csv_path = Path("perturbation_recall_rxrx3c.csv")
summary.to_csv(csv_path, index=False)
print(f"Saved: {csv_path}  ({len(summary)} rows)")

summary.set_index(["group", "view", "embedding"])

## Plot

In [ ]:
ks = list(range(1, K_MAX + 1))
n_groups = len(group_results)

for embedding_key in EMBEDDING_KEYS:
    tag = "pre_harmony" if embedding_key == "X_pca" else "post_harmony"
    # Per-figure y-max: pad 10%, round up to nearest 0.05, clamp to [0.1, 1.0].
    y_vals = [
        key_results[embedding_key][f"R@{k}"].mean()
        for view_results in group_results.values()
        for key_results in view_results.values()
        for k in ks
    ]
    y_max = float(np.clip(np.ceil(max(y_vals) * 1.1 * 20) / 20, 0.1, 1.0))

    fig, axes = plt.subplots(1, n_groups, figsize=(5 * n_groups, 4), sharey=True, squeeze=False)
    for ax, (group_name, view_results) in zip(axes[0], group_results.items()):
        for view, key_results in view_results.items():
            df = key_results[embedding_key]
            means = [df[f"R@{k}"].mean() for k in ks]
            ax.plot(ks, means, marker="o", linewidth=2, label=view)
        ax.set_xlabel("k")
        ax.set_xticks(ks)
        ax.set_ylim(0, y_max)
        ax.set_title(group_name)
        ax.grid(alpha=0.3)
        ax.legend(title="view")
    axes[0, 0].set_ylabel("Recall@k (macro-averaged)")
    fig.suptitle(
        f"Held-out perturbation Recall@k (Rxrx3C) ({tag}) — query={HELD_OUT_VALUE}",
        fontsize=13,
        y=1.02,
    )
    plt.tight_layout()
    plt.savefig(f"perturbation_recall_at_k_rxrx3c_{tag}.png", dpi=150, bbox_inches="tight")
    plt.show()
